In [1]:
%matplotlib qt5
%load_ext autoreload
%autoreload 2
    
import SF_RugPeek as sf
from os import listdir

import numpy as np
import matplotlib.pyplot as plt

In [2]:
Sample = 'Mb'
Sample_concentration = 4e-5

directory = r"C:\Users\tedc4\Documents\Stopped_Flow_UV-VIS\25_\Processed"

listdir(directory+f'\{Sample}')

['40.0uM_Mb_+10_equ-H2O2_processed.dat',
 '40.0uM_Mb_+10_equ-H2O2_st_dev_arr.dat',
 '40.0uM_Mb_+20_equ-H2O2_processed.dat',
 '40.0uM_Mb_+20_equ-H2O2_st_dev_arr.dat',
 '40.0uM_Mb_+30_equ-H2O2_processed.dat',
 '40.0uM_Mb_+30_equ-H2O2_st_dev_arr.dat',
 '40.0uM_Mb_+40_equ-H2O2_processed.dat',
 '40.0uM_Mb_+40_equ-H2O2_st_dev_arr.dat',
 '40.0uM_Mb_+50_equ-H2O2_processed.dat',
 '40.0uM_Mb_+50_equ-H2O2_st_dev_arr.dat',
 '40.0uM_Mb_+60_equ-H2O2_processed.dat',
 '40.0uM_Mb_+60_equ-H2O2_st_dev_arr.dat',
 '40.0uM_Mb_+70_equ-H2O2_processed.dat',
 '40.0uM_Mb_+70_equ-H2O2_st_dev_arr.dat',
 '40.0uM_Mb_+80_equ-H2O2_processed.dat',
 '40.0uM_Mb_+80_equ-H2O2_st_dev_arr.dat',
 '40.0uM_Mb_+90_equ-H2O2_processed.dat',
 '40.0uM_Mb_+90_equ-H2O2_st_dev_arr.dat']

In [5]:
sample_dict = {}
sig_dict = {}

sample_fnames = []

for i in range(len(listdir(directory+f'\{Sample}')) // 2):
    
    fname = listdir(directory+f'\{Sample}')[i*2][:listdir(directory+f'\{Sample}')[i*2].rindex('equ')+8]    
    sample_fnames.append(fname)
    
    sample_dict[fname] = sf.SF_Rug(directory=directory+f'\{Sample}', filename=str(listdir(directory+f'\{Sample}')[i*2]))
    sig_dict[f'{fname}_st_dev_arr'] = sf.SF_Rug(directory=directory+f'\{Sample}', filename=str(listdir(directory+f'\{Sample}')[(i*2)+1]))
    

In [7]:
## select a set of averaged scans loaded in as an SF_Rug object ##

file_idx = 2

sample = sample_dict[list(sample_dict)[file_idx]]
sample.sig_x =  sig_dict[list(sig_dict)[file_idx]].abs

sample_name = sample.fig_title.replace('_', ' ')

print(f'Selected file: {sample_name}')

Selected file: 40.0uM Mb +30 equ-H2O2


In [139]:
%autoreload 1
"""
inspect a plot of the ratio:
    sigma / |mean data| 
"""
a = np.zeros_like(sample.abs)

for idx, val in enumerate(a):
    for jdx, wal in enumerate(val):
        a[idx, jdx] = sample.sig_x[idx, jdx] / np.abs(sample.abs[idx, jdx]/1000)

vmin = np.nanmin(a)
vmax = np.nanmax(a)


fig, ax = plt.subplots()

im = ax.pcolormesh(sample.wavelengths,
                   sample.delays,
                   a,
                   cmap='Reds',
                   norm='log')  

tick_size = 15
axis_fontsize = 30
title_fontsize = 40
cbar_fontsize = 50

ax.tick_params(axis="both", labelsize = tick_size)




ax.set_xlabel('Wavelength (nm)', fontsize=axis_fontsize)
ax.xaxis.set_label_coords(0.53, -0.06) # (0, -0.05)

ax.set_ylabel('Delay (s)', fontsize=axis_fontsize, rotation=360)
ax.yaxis.set_label_coords(-0.1, 0.45) # (-0.08, 0.5)

ax.set_yscale('symlog')
#ax.set_ylim(-1, 10)

ax.set_title(sample.fig_title.replace('_', ' '), fontsize = title_fontsize, y=1.03)

tick_range = np.linspace(vmin, vmax, 10)
cbar = fig.colorbar(im) # , ticks=tick_range)

# cbar.set_label(label=r'$\frac{|{\Delta}O.D|}{\sigma}$', fontsize=cbar_fontsize, y = 0.5, labelpad = 80, rotation=360)
cbar.set_label(label=r'$\frac{\sigma}{|O.D|}$', fontsize=cbar_fontsize, y = 0.5, labelpad = 80, rotation=360)

cbar.ax.tick_params(labelsize = tick_size)

In [141]:
sample.explore_spectra()

In [ ]:
sample.explore_kinetics()

In [ ]:
## compute the SVD and generagte a plot of the eigenvalues and left and right eigenvectors ##
plt.close('all')
sample.compute_SVD(threshold=5000)
# sample.explore_SVD()

In [9]:
## construct the transfer matrix E ##

E = np.array([[0, 1],
              [0, 1]])

## specify the idx (in evals) of any rate coefficients for decay pathways that populate stable, absorbing products ##
Ep = None # [-1] 

## input the ratio of the initial compartment populations ##
C0 = np.array([1, 0]).T

# sigma = 1

## multiple c in case one compartment degrades to a spectroscopically inactive compartment i.e. from peroxide degradation
c_ll = np.array([])
c_ul = np.array([])

c_lims = (c_ll, c_ul)


In [11]:
## list the contents of the directory containing the DNS output file ##
save_directory = r"C:\Users\tedc4\Documents\Stopped_Flow_UV-VIS\25_\DNS_DATA"
modeltypes = ['\Competitive', '\Sequential', '\Parallel']
modeltype = modeltypes[1]

file_dir = listdir(save_directory+modeltype+f'\{Sample}')
file_dir

['1-cpmt-Sequential_model_80uM_hskmMb_+100_equ-H2O2_1000-linear-20s_DNS_0.save',
 '2-cpmt-Sequential_model_80uM_hskmMb_+10000_equ-H2O2_400-linear-2s_0c-vary=False_DNS_0.save',
 '2-cpmt-Sequential_model_80uM_hskmMb_+10000_equ-H2O2_DNS_0.save',
 '2-cpmt-Sequential_model_80uM_hskmMb_+100_equ-H2O2_1000-linear-100s_0c-vary=False_DNS_0.save',
 '2-cpmt-Sequential_model_80uM_hskmMb_+100_equ-H2O2_1000-linear-100s_0c-vary=False_DNS_1.save',
 '2-cpmt-Sequential_model_80uM_hskmMb_+100_equ-H2O2_1000-linear-20s_0c-vary=False_DNS_1.save',
 '2-cpmt-Sequential_model_80uM_hskmMb_+100_equ-H2O2_1000-linear-20s_1c-vary=False_DNS_0.save',
 '2-cpmt-Sequential_model_80uM_hskmMb_+100_equ-H2O2_1000-linear-20s_1c-vary=False_DNS_2.save',
 '2-cpmt-Sequential_model_80uM_hskmMb_+100_equ-H2O2_1000-linear-20s_DNS_0.save',
 '2-cpmt-Sequential_model_80uM_hskmMb_+100_equ-H2O2_1000-linear-20s_DNS_1.save',
 '2-cpmt-Sequential_model_80uM_hskmMb_+100_equ-H2O2_1000-linear-20s_DNS_2.save',
 '2-cpmt-Sequential_model_80uM_hskmMb

In [148]:
k_coef = []

In [13]:
## load saved DNS data ##
file_index = -5

vary_c = False

sample.load_DNS(E=E, C0=C0, Ep=Ep, c_lims=c_lims, var_c=vary_c, fname=f'{save_directory}{modeltype}\{Sample}\{file_dir[file_index]}')

print(f'Loaded: {file_dir[file_index]}\nDirectory: {save_directory}{modeltype}\{Sample}')
suffix = int(listdir(save_directory+modeltype+f'\{Sample}')[file_index][-6])
print(f'\nsuffix = {suffix}\n')

logz = sample.dres['logz'][-1]
logzerr = sample.dres['logzerr'][-1]

print(f'Mean parameters: {sample.DNS_fit_params}\nlogz: {logz} +/- {logzerr}')

Loaded: 2-cpmt_2-k_Ep=None_Sequential_model_40uM_Mb_+30_equ-H2O2_+0_equ-HS_1000-logarithmic-250s_0c-vary=False_DNS_0.save
Directory: C:\Users\tedc4\Documents\Stopped_Flow_UV-VIS\25_\DNS_DATA\Sequential\Mb

suffix = 0

Mean parameters: [  19.624      4930.43540332]
logz: -3.5968309190362304 +/- 0.036137204830926604


In [227]:
# k_coef.append(sample.DNS_fit_params[0])
# print(k_coef)

[21.54669218716359, 28.5450582290789, 19.623999995937258]


In [169]:
from scipy.optimize import curve_fit

In [229]:
def best_line(x, m, c):
    return (m*x) + c

p, cov = curve_fit(best_line, conc[0:len(k_coef)]/1000, k_coef)

In [231]:
p

array([-2403.36523906,    25.16127566])

In [236]:
## plot the rate coefficients against the concentration of hydrogen peroxide ##

conc = np.linspace(0.4, 3.6, 9) ## [H_{2}O_{2}] (mM) ##

x = np.linspace(10, 90, 9) ## equ. H_{2}O_{2} ##

K = '%#.3g' % p[0]

plt.close('all')

fig, ax = plt.subplots()

# ax.plot(conc[0:len(k_coef)], best_line(conc[0:len(k_coef)]/1000, *p), color='black', label=r'K$_{obs}$ = '+K+r' (M$^{-1}$ s$^{-1}$)')
# ax.scatter(conc[0:len(k_coef)], k_coef)

ax.plot(x[0:len(k_coef)], best_line(conc[0:len(k_coef)]/1000, *p), color='black', label=r'K$_{obs}$ = '+K+r' (M$^{-1}$ s$^{-1}$)')
ax.scatter(x[0:len(k_coef)], k_coef)

# ax.set_xlabel(r'[H$_{2}$O$_{2}$] (mM)')
ax.set_xlabel(r'equ. H$_{2}$O$_{2}$')

ax.set_ylabel(r'K$_{obs}$ (s$^{-1}$)')

ax.legend()


In [15]:
%autoreload 2
## plot the output from the nested sampler ##

# suffix = int(listdir(save_directory+modeltype+f'\{Sample}')[file_index][-6])
fig_directory = r"C:\Users\tedc4\Documents\Stopped_Flow_UV-VIS\25_\DNS-FIGURES-PPT"


## specify tau labels to indicate which compartments each decay component transfers molecules between ##
## APX-Fe[III] + (n)H2O2
# labelstau = [r'$\tau_{-Fe_{IV}-^{*+}}$', r'$\tau_{Fe_{IV}*}$', r'$\tau_{Fe_{III}*}$']

## APX-Fe[III] + (n)L-Asc + (n)H2O2
# labelstau = [r'$\tau_{-Fe_{IV}-^{*+}}$', r'$\tau_{Fe_{IV}}$', r'$\tau_{Fe_{III}}$']

## Mb-Fe[III] ##
labelstau = [r'$\tau_{Fe_{III}}$', r'$\tau_{Fe_{IV}}$']

## CcP-Fe[III] + (n)H2O2
# labelstau = [r'$\tau_{-Fe_{IV}-^{*+}}$', r'$\tau_{Fe_{IV}}$'] # , r'$\tau_{Fe_{III}}$']



## APX-Fe[III] + (n)H2O2
# e_labels = [r'$Fe_{III}*$', r'$Fe_{IV}*$', r'$-Fe_{IV}-^{*+}$']

## APX-Fe[III] + (n)L-Asc + (n)H2O2
# e_labels = [r'$Fe_{III}$', r'$Fe_{IV}$', r'$-Fe_{IV}-^{*+}$']

## Mb-Fe[III] ##
e_labels = [r'$Fe_{IV}$', r'$Fe_{III}$']

## CcP-Fe[III] + (n)H2O2
# e_labels = [r'$\tau_{Fe_{IV}}$', r'$\tau_{-Fe_{IV}-^{*+}}$']

# span = [0.999999426697, 1-1e-100, 0.999999426697, 0.999999426697]

plt.close('all')

sample.filename = sample.fig_title
filename = sample.fig_title.replace('.0', '')

plot = True
save_plots = False



fname = f'{E.shape[0]}-cpmt_{np.count_nonzero(E)}-k_Ep={Ep}_{modeltype[1:]}_model_{filename}_{sample.delays.shape[0]}-{sample.metadata[-1]}-{int(sample.delays[-1])}s_{c_lims[0].shape[0]}c-vary={vary_c}'


sf.SF_Rug.plot_nested_sampling(sample,
                               E=E,
                               C0=C0,
                               Ep=Ep,
                               c_lims=c_lims,
                               var_c=vary_c,
                               df_idx=None,
                               modelname=f'{E.shape[0]} Compartment {np.count_nonzero(E)}K {modeltype[1:]} model\n'+r'E$_{p}$'+f' = {Ep}',
                               fname=fname,
                               labelstau=labelstau,
                               e_labels=e_labels,
                               # span=None,
    
                               alignment='vertical',
                               # alignment='horizontal',
    
                               plot_traceplot=True,  # True
                               plot_runplot=True,
                               plot_cornerplot=plot, # True,
                               plot_cornerpoints=False,
                               plot_SF=plot, 
                               plot_SF_tileplot=plot,
                               # figsize=8,
    
                               save_dynesty_plots=save_plots,
                               save_SF_plots=save_plots,
                               save_corner_plots=save_plots,
                               save_SF_tileplot=save_plots,
                    
                               output_directory=f'{fig_directory}\{Sample}',
                               save_dpi=200,
                               title_fontsize=25,
                               label_fontsize=25,
                               tick_size = 10,
                               ud_idx=500,
                               SF_cmap='Reds')



In [ ]:
sample.explore_spectra()

In [ ]:
sample.explore_traces()

In [17]:
sample.explore_DNS_spectra(e_labels=e_labels,
                           Sample_concentration=Sample_concentration)

In [19]:
sample.explore_DNS_traces(e_labels=e_labels,
                          lx_lim=-1e-0,
                          ux_lim=400)